In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

In [15]:
df = pd.read_csv("../data/keystroke_data.csv")

df = df.sort_values(by=["PARTICIPANT_ID", "PRESS_TIME"])

In [16]:
df["hold_time"] = df["RELEASE_TIME"] - df["PRESS_TIME"]

df["next_press"] = df.groupby("PARTICIPANT_ID")["PRESS_TIME"].shift(-1)
df["flight_time"] = df["next_press"] - df["RELEASE_TIME"]

df = df.dropna()

# Keep only valid values
df = df[(df["hold_time"] > 0) & (df["flight_time"] > 0)]

In [17]:
top_users = df["PARTICIPANT_ID"].value_counts().head(50).index
df = df[df["PARTICIPANT_ID"].isin(top_users)]

In [18]:
sequence_length = 30

sequences = []
labels = []

for user in df["PARTICIPANT_ID"].unique():
    user_df = df[df["PARTICIPANT_ID"] == user]
    
    user_data = user_df[["hold_time", "flight_time"]].values
    
    for i in range(len(user_data) - sequence_length):
        seq = user_data[i:i+sequence_length]
        
        sequences.append(seq)
        labels.append(user)

X = np.array(sequences)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (32336, 30, 2)
y shape: (32336,)


In [19]:
le = LabelEncoder()
y = le.fit_transform(y)

num_classes = len(set(y))
print("Classes:", num_classes)

Classes: 50


In [20]:
from sklearn.preprocessing import StandardScaler

# reshape for scaling
num_samples, seq_len, num_features = X.shape

X_reshaped = X.reshape(-1, num_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

X = X_scaled.reshape(num_samples, seq_len, num_features)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [23]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [24]:
model = LSTMModel(
    input_size=2,
    hidden_size=128,   # increased
    num_classes=num_classes
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
epochs = 30

batch_size = 64

for epoch in range(epochs):
    model.train()
    
    permutation = torch.randperm(X_train.size(0))
    
    epoch_loss = 0
    
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        
        batch_X = X_train[indices]
        batch_y = y_train[indices]
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 1289.6105
Epoch 2, Loss: 1065.0465
Epoch 3, Loss: 952.6518
Epoch 4, Loss: 876.1554
Epoch 5, Loss: 830.1206
Epoch 6, Loss: 772.6792
Epoch 7, Loss: 731.9532
Epoch 8, Loss: 697.7545
Epoch 9, Loss: 665.0705
Epoch 10, Loss: 635.5757
Epoch 11, Loss: 599.9259
Epoch 12, Loss: 576.7305
Epoch 13, Loss: 551.2122
Epoch 14, Loss: 525.0667
Epoch 15, Loss: 498.5413
Epoch 16, Loss: 471.0710
Epoch 17, Loss: 449.0903
Epoch 18, Loss: 429.1600
Epoch 19, Loss: 406.1997
Epoch 20, Loss: 377.3110
Epoch 21, Loss: 360.5585
Epoch 22, Loss: 334.6214
Epoch 23, Loss: 310.1115
Epoch 24, Loss: 291.9565
Epoch 25, Loss: 269.1215
Epoch 26, Loss: 255.4984
Epoch 27, Loss: 227.4460
Epoch 28, Loss: 214.4208
Epoch 29, Loss: 210.4320
Epoch 30, Loss: 194.8070


In [26]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    
    acc = (predicted == y_test).float().mean()
    
print(" LSTM Accuracy:", acc.item())

 LSTM Accuracy: 0.8511132001876831


In [27]:
# Check train vs test accuracy

model.eval()

with torch.no_grad():
    # Train accuracy
    train_pred = model(X_train).argmax(1)
    train_acc = (train_pred == y_train).float().mean()

    # Test accuracy
    test_pred = model(X_test).argmax(1)
    test_acc = (test_pred == y_test).float().mean()

print("Train Accuracy:", train_acc.item())
print("Test Accuracy:", test_acc.item())

Train Accuracy: 0.8984072804450989
Test Accuracy: 0.8511132001876831


In [28]:
import torch

# Save model
torch.save(model.state_dict(), "../models/lstm_model.pth")

print("✅ LSTM model saved successfully")

✅ LSTM model saved successfully


In [29]:
import pickle

with open("../models/lstm_label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ Label encoder saved")

✅ Label encoder saved
